# AML Detection with a Simple GAT
This notebook runs the same small-graph pipeline as `train.py`. It uses up to 25,000 rows from the IBM `HI-Small_Trans.csv` file.

**Target:** account-level proxy label derived from `Is Laundering`. This is an educational baseline, not a production AML decision system.

## 1. Install dependencies
Run once in a fresh Codespace:
```python
%pip install -r requirements.txt
```

In [ ]:
import torch
import matplotlib.pyplot as plt
from dataset import build_graph
from train import make_masks, evaluate
from model import AMLGAT
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 2. Download data and build a small graph
Kaggle access must be configured in Codespaces. The graph builder reads at most 25,000 rows to keep the experiment manageable.

In [ ]:
data = build_graph(max_transactions=25000, seed=42)
data = make_masks(data, seed=42)
print(data)
print('Node features:', data.x.shape)
print('Edge features:', data.edge_attr.shape)
print('Positive accounts:', int(data.y.sum()))

## 3. Train and evaluate
For the recommended complete run, execute `python train.py` in the Codespaces terminal. The code below is a compact inspection example; `train.py` includes early stopping, checkpointing, metrics, and plots.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)
model = AMLGAT(data.x.size(1), data.edge_attr.size(1)).to(device)
logits = model(data.x, data.edge_index, data.edge_attr)
print('Logits shape:', logits.shape)
print('Example probabilities:', torch.sigmoid(logits[:10]).detach().cpu())

## 4. Run the full experiment
In a terminal:
```bash
python train.py
```

Review `artifacts/metrics.json`, `artifacts/training_loss.png`, and `artifacts/validation_pr_auc.png`. The final test metrics are produced after selecting the best checkpoint using validation PR-AUC.

## 5. Interpretation and limitations
- PR-AUC is especially useful when positive accounts are rare.
- Precision and recall depend on the selected probability threshold.
- The account label is positive when an endpoint appears in a labelled laundering transaction in the sampled data.
- Node aggregates use the sampled period, and the graph includes all sampled edges. This is a transductive baseline, not a realistic future-time evaluation.
- A future version should use a chronological split and build node features using historical transactions only.